In [0]:
%run ../../02_common_utils/operations

In [0]:
from datetime import datetime

team_name  = "team_lemma"
bronze_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.bronze"
silver_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.silver"
staging_db = f"charles_schwab_retailbrokerage_dev_{team_name}.staging"

spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
from pyspark.sql.functions import col, trim, to_date, current_timestamp, lit
from pyspark.sql.types import DecimalType, LongType

In [0]:


dbutils.widgets.dropdown("current_batch", "2", ["2", "3"], "Current Batch")
current_batch = dbutils.widgets.get("current_batch")
batch_label   = f"Batch{current_batch}"
try:
    run_info_row = spark.sql(f"SELECT _run_id, _batch FROM {staging_db}.dailymarket_current LIMIT 1").first()
    carried_run_id = run_info_row[0] if run_info_row else "unknown"
    carried_batch = run_info_row[1] if run_info_row else batch_label
except Exception:
    carried_run_id = "unknown"
    carried_batch = batch_label

run_id=carried_run_id
print(f"carried_run_id: {carried_run_id}")
print(f"carried_batch: {carried_batch}")
print(f"Merging staging.dailymarket_current → silver.markethistory for {batch_label}")

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'staging_to_sliver_market_dailymarket_2_3', f'Starting processing for DailyMarket CDC to Silver (batch: {carried_batch})')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'RUNNING')

In [0]:
from delta.tables import DeltaTable

batch_label = f"Batch{current_batch}"

staged = (
    spark.table(f"{staging_db}.dailymarket_current")
    .filter(col("cdc_action").isin("N", "C", "D"))
)

silver_tbl = DeltaTable.forName(
    spark,
    f"charles_schwab_retailbrokerage_dev_{team_name}.silver.markethistory"
)

(
    silver_tbl.alias("s")
    .merge(
        staged.alias("i"),
        
        "s.dm_date = TRY_TO_DATE(TRIM(i.DM_DATE), 'yyyy-MM-dd') "
        "AND s.dm_s_symb = TRIM(i.DM_S_SYMB)"
    )
    .whenMatchedUpdate(
        condition = "i.cdc_action = 'C'",
        set = {
            "dm_close" : "CAST(TRIM(i.DM_CLOSE) AS DECIMAL(8,2))",
            "dm_high"  : "CAST(TRIM(i.DM_HIGH)  AS DECIMAL(8,2))",
            "dm_low"   : "CAST(TRIM(i.DM_LOW)   AS DECIMAL(8,2))",
            "dm_vol"   : "CAST(TRIM(i.DM_VOL)   AS BIGINT)",
            "_batch"   : f"'{batch_label}'",
            "_run_id"  : f"'{run_id}'",
            "_load_ts" : "current_timestamp()",
        }
    )
    .whenMatchedDelete(
        condition = "i.cdc_action = 'D'"
    )
    .whenNotMatchedInsert(
        condition = "i.cdc_action = 'N'",
        values = {
            "dm_date"  : "TRY_TO_DATE(TRIM(i.DM_DATE), 'yyyy-MM-dd')",
            "dm_s_symb": "TRIM(i.DM_S_SYMB)",
            "dm_close" : "CAST(TRIM(i.DM_CLOSE) AS DECIMAL(8,2))",
            "dm_high"  : "CAST(TRIM(i.DM_HIGH)  AS DECIMAL(8,2))",
            "dm_low"   : "CAST(TRIM(i.DM_LOW)   AS DECIMAL(8,2))",
            "dm_vol"   : "CAST(TRIM(i.DM_VOL)   AS BIGINT)",
            "_batch"   : f"'{batch_label}'",
            "_run_id"  : f"'{run_id}'",
            "_load_ts" : "current_timestamp()",
        }
    )
    .execute()
)

total = spark.table(f"{silver_db}.markethistory").count()
print(f" silver.markethistory after {batch_label}: {total:,}")


In [0]:
checks = {
    "silver.markethistory" : (spark.table(f"{silver_db}.markethistory").count(),  5_285_024)
}



print(f"\n{'Table':<25} {'Actual':>10} {'Expected':>10} {'Status'}")
print("-" * 55)
for tbl, (actual, expected) in checks.items():
    status = "PASS" if actual == expected else " FAIL"
    print(f"{tbl:<25} {actual:>10,} {expected:>10,}  {status}")

In [0]:

source_count = (
    spark.table(f"{bronze_db}.dailymarket")
    .filter(col("_batch") == "Batch1")
    .count()
)
target_count = (
    spark.table(f"{silver_db}.markethistory")
    .filter(col("_batch") == "Batch1")
    .count()
)
carried_run_id = str(
    spark.table(f"{silver_db}.markethistory")
    .filter(col("_batch") == "Batch1")
    .select("`_run_id`").first()[0]
)

log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id="Batch1",
    domain="MARKET",
    table_name="markethistory",
    source_layer="bronze",
    target_layer="silver",
    source_count=source_count,
    target_count=target_count
)

log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch="Batch1",
    layer="silver",
    table_name="markethistory",
    operation="OVERWRITE",
    rows_affected=target_count
)

print(f"source_count : {source_count:,}")   
print(f"target_count : {target_count:,}")   

In [0]:
staging_df = spark.table(f"{staging_db}.dailymarket_current")

source_count = staging_df.filter(col("cdc_action").isin("N", "C")).count()


target_count = (
    spark.table(f"{silver_db}.markethistory")
    .filter(col("_batch") == batch_label)
    .count()
)

carried_run_id = str(staging_df.select("`_run_id`").first()[0])

log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id=batch_label,
    domain="MARKET",
    table_name="markethistory",
    source_layer="staging",
    target_layer="silver",
    source_count=source_count,
    target_count=target_count
)

log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch=batch_label,
    layer="silver",
    table_name="markethistory",
    operation="MERGE",
    rows_affected=source_count
)


log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'staging_to_sliver_market_dailymarket_2_3', 'Successfully completed DailyMarket CDC processing to silver layer.')
print(f"source_count (N+C only) : {source_count:,}")
print(f"target_count (this batch): {target_count:,}")

